# Examine Fusion

Analyses the fusion evaporation from the beam and target. 

1) Load from the configuration file the properties of the experiment, and get the beam energy range inside the AT-TPC.

In [ ]:
# Ideally, you should not edit the next cell
experiment_tag = "exp" # indicate the name of the configuration file

In [ ]:
# NOTE FOR USER: You can collapse this cell, extracts data from experiment tag
# Personal NOTE: this could be a single function, and create global variables
import yaml 
from pathlib import Path
from spyral_utils.nuclear import NuclearDataMap
import numpy as np 
import polars as pl
import matplotlib.pyplot as plt 
from spyral_utils.plot.histogram import Histogrammer
import matplotlib as mpl
cmap = mpl.colormaps["rainbow"]
%matplotlib widget

# This is for the meantime -------------------------------------------------------
# The final motivation is to make it pip installeable 
import sys
src_path = Path.cwd().parents[1]
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))
# --------------------------------------------------------------------------------
from attpc_analysis.src.detector_settings.energy_loss_setup import EnergyLossSetup
from attpc_analysis.src.data_models.threebodydecay import dalitz_plot
from attpc_analysis.src.load.io import RunSolverDataLoader
from attpc_analysis.src.data_models.physics import DataReactionKinematics

nuclear_map = NuclearDataMap()

with open(f"./config/{experiment_tag}.yaml","r") as file: 
    config = yaml.safe_load(file)


# Import variables: paths, runs, experiments details, and detector settings,  
## paths to folders 
paths_to_experiment = config["paths_to_experiment"]
workspace_path = Path(paths_to_experiment["workspace_path"])
trace_path = Path(paths_to_experiment["trace_path"])
target_material_path = Path(paths_to_experiment["target_path"])

solver_result_path = workspace_path/ "InterpSolver"

## run details 
runs_details = config["runs"]
run_min = runs_details["run_min"]
run_max = runs_details["run_max"]
runs_to_skip = runs_details["runs_to_skip"]

## detector settings
detector_settings = config["detector_settings"]
ic_pressure = detector_settings["ic_pressure"]

## experiment details 
experiment_details = config["experiment_details"]
beam_energy = experiment_details ["beam_energy"]

### properties of each nuclear component 
beam_z, beam_a = experiment_details["beam"]
target_z, target_a = experiment_details["target"]
targetlike_z, targetlike_a =experiment_details["targetlike"]
beamlike_z, beamlike_a = beam_z+target_z-targetlike_z, beam_a + target_a - targetlike_a

beam = nuclear_map.get_data(beam_z, beam_a)
target = nuclear_map.get_data(target_z, target_a)
beamlike = nuclear_map.get_data(beamlike_z, beamlike_a)
targetlike = nuclear_map.get_data(targetlike_z, targetlike_a)


# Get the beam energy for the active area of the detector
energy_loss_setup = EnergyLossSetup(
    beam_energy = beam_energy,
    projectile = beam,
    attpc_target_path_json= target_material_path,
    ic_gas_pressure=600
)

target_material = energy_loss_setup.active_gas_target_material()
proj_energy_start = energy_loss_setup.beam_energy_entrance_attpc()
proj_energy_stop = energy_loss_setup.beam_energy_exit_attpc()

print(f"Reaction: {target}({beam}, {beamlike}){targetlike}")
print(f"Target material: {target_material.ugly_string}")
print(f"Accelerator Beam energy: {beam_energy} MeV")
print(f"Beam energy range in AT-TPC: {proj_energy_start}-{proj_energy_stop} MeV")

### Fusion events for 3-body decay with one track per particle
2. Input the three particles you would like to analyze for fusion

In [ ]:
particle_1_z, particle_1_a = 2, 4 
particle_2_z, particle_2_a = 1, 1
particle_3_z, particle_3_a = 1, 2 

In [ ]:
# Construct nuclear properties 
particle_1 = nuclear_map.get_data(particle_1_z,particle_1_a)
particle_2 = nuclear_map.get_data(particle_2_z,particle_2_a)
particle_3 = nuclear_map.get_data(particle_3_z,particle_3_a)
# create compound nucleus
compound_nucleus_z = particle_1_z + particle_2_z + particle_3_z
compound_nucleus_a = particle_1_a + particle_2_a + particle_3_a
compound_nucleus = nuclear_map.get_data(compound_nucleus_z, compound_nucleus_a)

#get residuals
residual_1 = nuclear_map.get_data(beam_z + target_z - particle_1_z, beam_a + target_a - particle_1_a)
residual_2 = nuclear_map.get_data(beam_z + target_z - particle_2_z, beam_a + target_a - particle_2_a)
residual_3 = nuclear_map.get_data(beam_z + target_z - particle_3_z, beam_a + target_a - particle_3_a)

results = {}
results["m12"] = np.empty(0,float)
results["m13"] = np.empty(0,float)
results["m23"] = np.empty(0,float)
results["fusion_ex"]= np.empty(0,float)
results["event"] = np.empty(0,int)
results["run"] = np.empty(0,int)

# Store Physics into dictations 
dictations_items = ["brho","kinetic_energy", "polar", "az", "redchisq", "vertex_x", "vertex_y", "vertex_z","ex","polar_cm"]

kinematics_p1 = {}
kinematics_p2 = {}
kinematics_p3 = {}
for items in dictations_items:
    kinematics_p1[items] = np.empty(0,float)
    kinematics_p2[items] = np.empty(0,float)
    kinematics_p3[items] = np.empty(0,float)

# Load particle 2 particle file 
empty_files = []
data_p1 = RunSolverDataLoader(
    solver_result_path=solver_result_path,
    particle = particle_1,
    empty_files=empty_files
)
data_p2 = RunSolverDataLoader(
    solver_result_path,
    particle_2, 
    empty_files
)
data_p3 = RunSolverDataLoader(
    solver_result_path, 
    particle_3,
    empty_files
)

for run in range(run_min, run_max): 
    # Open data frames 
    df_p1 = data_p1.load_solver(run)
    df_p2 = data_p2.load_solver(run)
    df_p3 = data_p3.load_solver(run)
    if empty_files.__contains__(run):
        continue 
    #------------------------------------------------------------------------------
    # Insert specific filters into the dataframe 
    # (Here)
    # select events inside the gates 
    events_p1, counts_p1 = np.unique(df_p1.select("orig_event").to_numpy().flatten(),return_counts=True)
    events_p1 = events_p1[counts_p1==1]

    events_p2, counts_p2 = np.unique(df_p2.select("orig_event").to_numpy().flatten(),return_counts=True)
    events_p2 = events_p2[counts_p2==1]

    events_p3, counts_p3 = np.unique(df_p3.select("orig_event").to_numpy().flatten(),return_counts=True)
    events_p3 = events_p3[counts_p3==1]

    events = np.intersect1d(events_p1,events_p2)
    events = np.intersect1d(events,events_p3)

    df_p1 = df_p1.filter(pl.col("orig_event").is_in(events))
    df_p2 = df_p2.filter(pl.col("orig_event").is_in(events))
    df_p3 = df_p3.filter(pl.col("orig_event").is_in(events))
    #------------------------------------------------------------------------------
    # Extract reactions physics
    reaction_p1 = DataReactionKinematics(
        data=df_p1,
        projectile=beam,
        target=target,
        targetlike=particle_1,
        beamlike=residual_1, 
        beam_energy_entrace_tpc=proj_energy_start
    )
    reaction_p2 = DataReactionKinematics(df_p2, beam, target,particle_2, residual_2, proj_energy_start)
    reaction_p3 = DataReactionKinematics(df_p3, beam, target,particle_3, residual_3, proj_energy_start)

    #close data frames 
    df_p1, df_p2, df_p3 = None, None, None

    # construct fusion vectors
    particle_1_vector = reaction_p1.construct_targetlike_vector()
    particle_2_vector = reaction_p2.construct_targetlike_vector()
    particle_3_vector = reaction_p3.construct_targetlike_vector()

    m12 = (particle_1_vector + particle_2_vector).M
    m13 = (particle_1_vector + particle_3_vector).M
    m23 = (particle_2_vector + particle_3_vector).M
    fusion_ex = (particle_1_vector+particle_2_vector+particle_3_vector).mass - compound_nucleus.mass

    # Store information 
    results["m12"] = np.append(results["m12"], m12)
    results["m13"] = np.append(results["m13"], m13)
    results["m23"] = np.append(results["m23"], m23)
    results["fusion_ex"] = np.append(results["fusion_ex"],fusion_ex)
    results["event"] = np.append(results["event"],events)
    results["run"] = np.append(results["run"],np.full(len(events),run))
    
    # Extract Physics into dict
    reaction_p1.extract_kinematics_values(kinematics_p1,target_material)
    reaction_p2.extract_kinematics_values(kinematics_p2,target_material)
    reaction_p3.extract_kinematics_values(kinematics_p3,target_material)



# remove duplicates
empty_files = sorted(list(set(empty_files)))
print("Runs without data of interest: ", empty_files)

#### Multiple Plots 
Dalitz plot and then, excitation spectrum of the compound nucleus from fusion. 

In [ ]:
energy_threshold =  (particle_1.mass + particle_2.mass  + particle_3.mass ) - compound_nucleus.mass
print("energy threshold", energy_threshold)

In [ ]:
# Binning 
min_bin, max_bin = -2 , 60
bin_size = 100
granularity = (max_bin - min_bin) / bin_size


# Product Plots
grammer = Histogrammer()
grammer.add_hist1d("fusion_ex", bin_size, (min_bin, max_bin))
grammer.fill_hist1d("fusion_ex", results["fusion_ex"][:])
hist = grammer.get_hist1d("fusion_ex")
fig, ax = plt.subplots(1,1)
ax.stairs(hist.counts, edges=hist.bins)
ax.set_title(f"{compound_nucleus.get_latex_rep()} Fusion Evaporation ", fontsize=16)
ax.set_xlabel(r"Energy Spectrum (MeV)", fontsize=16)
ax.set_ylabel(f"Counts/{int(granularity*1000)} keV", fontsize=16)
# ax.axvline(38.27, color='blue')#, label='98 MeV (before IC)')
# # ax.axvspan(35.41,36.71,color='blue',alpha=0.2,label='AT-TPC')

ax.axvline(energy_threshold,color='black',label='threshold')
ax.legend()

In [ ]:
# mask = (kinematics_p2["ex"]>10) & (kinematics_p2["ex"] < 20)

# a + 3He + d 
# grammer.add_hist2d("inv_mass_2d",  (60, 60), ((6535, 6560), (5603, 5625)))
grammer.add_hist2d("inv_mass_2d",  (40, 40), ((6535, 6560), (5603, 5625)))

grammer.fill_hist2d("inv_mass_2d",results["m12"][:], results["m13"][:])
hist = grammer.get_hist2d("inv_mass_2d")
fig, ax = plt.subplots(1,1)
mesh = ax.pcolormesh(hist.x_bins, hist.y_bins, hist.counts)#, norm="log")

# Add resonances 
ax.axvline(nuclear_map.get_data(4,7).mass + 4.570,color='cyan',linewidth=3, label=r'$E_x(^7\text{Be})=4.57$ MeV')
ax.axhline((nuclear_map.get_data(3,6).mass + 2.186),color='red',linewidth=3,label=r'$E_x(^6\text{Li})=2.186$ MeV')

# Lower limit
dalitz_allowed = dalitz_plot(
    m1 = particle_1,
    m2 = particle_2,
    m3 = particle_3,
    compound = compound_nucleus,
    ex = 35.41
)

x, y_min, y_max = dalitz_allowed.values_plot(steps=1000)
# ax.plot(np.sqrt(x),np.sqrt(y_min),color='red',linewidth=3,label='exit AT-TPC')
# ax.plot(np.sqrt(x),np.sqrt(y_max),color='red',linewidth=3)


dalitz_allowed = dalitz_plot(
    m1 = particle_1,
    m2 = particle_2,
    m3 = particle_3,
    compound = compound_nucleus,
    ex = 36.71
)

x, y_min, y_max = dalitz_allowed.values_plot(steps=1000)
# ax.plot(np.sqrt(x),np.sqrt(y_min),color='blue',linewidth=3, label='entrance AT-TPC')
# ax.plot(np.sqrt(x),np.sqrt(y_max),color='blue',linewidth=3)


dalitz_allowed = dalitz_plot(
    m1 = particle_1,
    m2 = particle_2,
    m3 = particle_3,
    compound = compound_nucleus,
    ex = 39.6
)

x, y_min, y_max = dalitz_allowed.values_plot(steps=1000)
# ax.plot(np.sqrt(x),np.sqrt(y_min),color='magenta',linewidth=3, label='centroid')
# ax.plot(np.sqrt(x),np.sqrt(y_max),color='magenta',linewidth=3)

fig.colorbar(mesh, ax=ax)
ax.set_title(f"{compound_nucleus.get_latex_rep()} Fusion Evaporation ", fontsize=16)
ax.set_xlabel(f"m({particle_1.get_latex_rep()}, {particle_2.get_latex_rep()}) [MeV]", fontsize=16)
ax.set_ylabel(f"m({particle_1.get_latex_rep()}, {particle_3.get_latex_rep()}) [MeV]", fontsize=16)
fig.set_figheight(8.0)
fig.set_figwidth(11.0)
ax.legend(fontsize=16)

In [ ]:
# Save events information 
frame = {"run": results["run"], "event": results["event"]}
df = pl.DataFrame(frame)
# df.write_csv("./fusion_events.csv")

In [ ]:
# Make kinematics plots
bins = 100
ngrid = 1000 # grid for kinematic plot 
make_cuts = False 

residual_excitations_p1 = np.array([20.53,23.74,25.44,25.44]) #5Li
residual_excitations_p2 = np.array([2.186]) #6Li
residual_excitations_p3 = np.array([4.57]) #6Be



# Execute plots
plt.close("all")
fig, axs = plt.subplots(1,3,figsize=(16,6))

# mask =  results["m12"] < 6542 # 7Be
# mask =  results["m13"] < 5605 # 6Li
# mask =  (results["m12"] > 6542) & (results["m13"] > 5605)
# mask = (results["m12"] > 6542) & (kinematics_p3["kinetic_energy"]>10)# Remove not 7Be inelastic channels

# mask = (results["m12"] < 6545) & (kinematics_p1["kinetic_energy"]>40) & (kinematics_p2["kinetic_energy"]>30)

if not make_cuts:
    mask = np.full(len(results["m12"]),True)


ax = axs[0]
grammer.add_hist2d("kinematics_p1", (bins,bins), ((0,90),(0,70)))
grammer.fill_hist2d("kinematics_p1", np.rad2deg(kinematics_p1["polar"][mask]), kinematics_p1["kinetic_energy"][mask])
hist = grammer.get_hist2d("kinematics_p1")
mesh = ax.pcolormesh(hist.x_bins, hist.y_bins, hist.counts, norm='log')
ax.set_title(f"Kinematics of {particle_1.get_latex_rep()}")
ax.set_xlabel(r"$\theta_\text{lab}$ (deg)")
ax.set_ylabel("Kinetic energy (MeV)")


eject_kinematics = np.zeros((len(residual_excitations_p1), ngrid * 2, 2))
eject_kinematics[:, :ngrid, :] = reaction_p1.kinematic_curve(proj_energy_start,residual_excitations_p1)
eject_kinematics[:, ngrid:, :] = np.flip(reaction_p1.kinematic_curve(proj_energy_stop, residual_excitations_p1),axis=1)

for idx, ex in enumerate(residual_excitations_p1):
    ax.plot(np.rad2deg(eject_kinematics[idx, :1000, 0]), eject_kinematics[idx, :1000, 1], label=fr"$E_x$({residual_1.get_latex_rep()})={ex:0.2f} MeV")
    ax.fill(np.rad2deg(eject_kinematics[idx, :, 0]), eject_kinematics[idx, :, 1],alpha=0.2)#, label=f"Ex={ex:02} Band")

ax.legend()

ax = axs[1]
grammer.add_hist2d("kinematics_p2", (bins,bins), ((0,90),(0,70)))
grammer.fill_hist2d("kinematics_p2", np.rad2deg(kinematics_p2["polar"][mask]), kinematics_p2["kinetic_energy"][mask])
hist = grammer.get_hist2d("kinematics_p2")
mesh = ax.pcolormesh(hist.x_bins, hist.y_bins, hist.counts, norm='log')
ax.set_title(f"Kinematics of {particle_2.get_latex_rep()}")
ax.set_xlabel(r"$\theta_\text{lab}$ (deg)")
ax.set_ylabel("Kinetic energy (MeV)")


eject_kinematics = np.zeros((len(residual_excitations_p2), ngrid * 2, 2))
eject_kinematics[:, :ngrid, :] = reaction_p2.kinematic_curve(proj_energy_start,residual_excitations_p2)
eject_kinematics[:, ngrid:, :] = np.flip(reaction_p2.kinematic_curve(proj_energy_stop, residual_excitations_p2),axis=1)

for idx, ex in enumerate(residual_excitations_p2):
    ax.plot(np.rad2deg(eject_kinematics[idx, :ngrid, 0]), eject_kinematics[idx, :ngrid, 1], label=fr"$E_x$({residual_2.get_latex_rep()})= {ex:0.2f} MeV")#,color=colors[idx])
    ax.fill(np.rad2deg(eject_kinematics[idx, :, 0]), eject_kinematics[idx, :, 1],alpha=0.2)#, color=colors[idx],)#, label=f"Ex={ex:02} Band", alpha=0.2)

ax.legend()


ax = axs[2]
grammer.add_hist2d("kinematics_p3", (bins,bins), ((0,90),(0,70)))
grammer.fill_hist2d("kinematics_p3", np.rad2deg(kinematics_p3["polar"][mask]), kinematics_p3["kinetic_energy"][mask])
hist = grammer.get_hist2d("kinematics_p3")
mesh = ax.pcolormesh(hist.x_bins, hist.y_bins, hist.counts, norm='log')
ax.set_title(f"Kinematics of {particle_3.get_latex_rep()}")
ax.set_xlabel(r"$\theta_\text{lab}$ (deg)")
ax.set_ylabel("Kinetic energy (MeV)")



eject_kinematics = np.zeros((len(residual_excitations_p3), ngrid * 2, 2))
eject_kinematics[:, :ngrid, :] = reaction_p3.kinematic_curve(proj_energy_start,residual_excitations_p3)
eject_kinematics[:, ngrid:, :] = np.flip(reaction_p3.kinematic_curve(proj_energy_stop, residual_excitations_p3),axis=1)

for idx, ex in enumerate(residual_excitations_p3):
    ax.plot(np.rad2deg(eject_kinematics[idx, :1000, 0]), eject_kinematics[idx, :1000, 1], label=fr"$E_x$({residual_3.get_latex_rep()})={ex:02} MeV")#,color=colors[idx])
    ax.fill(np.rad2deg(eject_kinematics[idx, :, 0]), eject_kinematics[idx, :, 1],alpha=0.2)#, color=colors[idx],)#, label=f"Ex={ex:02} Band", alpha=0.2)

ax.legend()


In [ ]:
# Save events information 
frame = {"run": results["run"][mask], "event": results["event"][mask]}
df = pl.DataFrame(frame)
# df.write_csv("./fusion_9B_without_7Be_resonance.csv")

In [ ]:
# Plot Particle 1 Excitation Spectrums with Invariant Mass and Missing Mass
# Option 1
min_bin, max_bin = 14, 35 
granularity = 0.25 # MeV
bin_size = int((max_bin - min_bin) / (granularity))
# Option 2
# bin_size = 100
# granularity = (max_bin - min_bin) / bin_size



fig, axs = plt.subplots(1,2, figsize=(16,6))
eject_kinematics = np.zeros((len(residual_excitations_p1), ngrid * 2, 2))
eject_kinematics[:, :ngrid, :] = reaction_p1.kinematic_curve(proj_energy_start,residual_excitations_p1)
eject_kinematics[:, ngrid:, :] = np.flip(reaction_p1.kinematic_curve(proj_energy_stop, residual_excitations_p1),axis=1)
colors = cmap(np.linspace(0,1,len(residual_excitations_p1)))

# Particle 1: Missing mass 
ax = axs[0]
ax.set_title("Missing Mass Technique")
grammer.add_hist1d("ex_1", bin_size, (min_bin,max_bin))
grammer.fill_hist1d("ex_1", kinematics_p1["ex"][mask])
hist  = grammer.get_hist1d("ex_1")
uncertainty = np.sqrt(np.maximum(hist.counts,1))
hist_x = hist.bins[:-1] + 0.5*hist.bin_width
#option 1
# ax.errorbar(hist_x, hist.counts, yerr=uncertainty, fmt='o', markersize=3, capsize=2,color='black', linestyle='none')#,label='missing')
#option 2
ax.errorbar(hist_x, hist.counts, yerr=uncertainty,  color='black', linestyle='none')
ax.stairs(hist.counts, edges=hist.bins, color='black', linewidth=1)
for idx, ex in enumerate(residual_excitations_p1):
    ax.axvline(ex,color=colors[idx],label=fr"$E_x$={ex:0.2f} MeV")


# Quick
ax.axvline(16.66, color='navy',label="S($^3$He)=16.66 MeV ", linestyle='--')
ax.set_ylabel(f"Counts/{int(granularity*1000)} keV")
ax.set_xlabel(f"{residual_1.get_latex_rep()} excitation energy (MeV)")
# ax.grid()


# Particle 1: Invariant mass 
ax = axs[1]
ax.set_title("Invariant Mass Technique")
grammer.add_hist1d("ex_1_inv", bin_size, (min_bin,max_bin))
grammer.fill_hist1d("ex_1_inv", results["m23"][mask] - residual_1.mass)
hist  = grammer.get_hist1d("ex_1_inv")
uncertainty = np.sqrt(np.maximum(hist.counts,1))
hist_x = hist.bins[:-1] + 0.5*hist.bin_width
#option 1
# ax.errorbar(hist_x, hist.counts, yerr=uncertainty, fmt='o', markersize=3, capsize=2,color='black', linestyle='none')#,label='missing')
#option 2
ax.errorbar(hist_x, hist.counts, yerr=uncertainty,  color='black', linestyle='none')
ax.stairs(hist.counts, edges=hist.bins, color='black', linewidth=1, label='invariant')


for idx, ex in enumerate(residual_excitations_p1):
    ax.axvline(ex,color=colors[idx])#, label=fr"$E_x$={ex:0.2f} MeV",)



# Quick
ax.axvline(16.66, color='navy', linestyle='--')#,label="S($^3$He)=16.66 MeV ")
ax.set_ylabel(f"Counts/{int(granularity*1000)} keV")
ax.set_xlabel(f"{residual_1.get_latex_rep()} excitation energy (MeV)")
# ax.grid()
fig.legend()


In [ ]:
# Plot Particle 2 Excitation Spectrums with Invariant Mass and Missing Mass
# Option 1
min_bin, max_bin = -2 , 20
granularity = 0.25 # MeV
bin_size = int((max_bin - min_bin) / (granularity))
# Option 2
# bin_size = 100
# granularity = (max_bin - min_bin) / bin_size



fig, axs = plt.subplots(1,2, figsize=(16,6))
eject_kinematics = np.zeros((len(residual_excitations_p2), ngrid * 2, 2))
eject_kinematics[:, :ngrid, :] = reaction_p2.kinematic_curve(proj_energy_start,residual_excitations_p2)
eject_kinematics[:, ngrid:, :] = np.flip(reaction_p2.kinematic_curve(proj_energy_stop, residual_excitations_p2),axis=1)
colors = cmap(np.linspace(0,1,len(residual_excitations_p2)))

# Particle 1: Missing mass 
ax = axs[0]
ax.set_title("Missing Mass Technique")
grammer.add_hist1d("ex_2", bin_size, (min_bin,max_bin))
grammer.fill_hist1d("ex_2", kinematics_p2["ex"][mask])
hist  = grammer.get_hist1d("ex_2")
uncertainty = np.sqrt(np.maximum(hist.counts,1))
hist_x = hist.bins[:-1] + 0.5*hist.bin_width
#option 1
# ax.errorbar(hist_x, hist.counts, yerr=uncertainty, fmt='o', markersize=3, capsize=2,color='black', linestyle='none')#,label='missing')
#option 2
ax.errorbar(hist_x, hist.counts, yerr=uncertainty,  color='black', linestyle='none')
ax.stairs(hist.counts, edges=hist.bins, color='black', linewidth=1)
for idx, ex in enumerate(residual_excitations_p2):
    ax.axvline(ex,color=colors[idx],label=fr"$E_x$={ex:0.2f} MeV")


# Quick
ax.set_ylabel(f"Counts/{int(granularity*1000)} keV")
ax.set_xlabel(f"{residual_2.get_latex_rep()} excitation energy (MeV)")
# ax.grid()


# Particle 1: Invariant mass 
ax = axs[1]
ax.set_title("Invariant Mass Technique")
grammer.add_hist1d("ex_2_inv", bin_size, (min_bin,max_bin))
grammer.fill_hist1d("ex_2_inv", results["m13"][mask] - residual_2.mass)
hist  = grammer.get_hist1d("ex_2_inv")
uncertainty = np.sqrt(np.maximum(hist.counts,1))
hist_x = hist.bins[:-1] + 0.5*hist.bin_width
#option 1
# ax.errorbar(hist_x, hist.counts, yerr=uncertainty, fmt='o', markersize=3, capsize=2,color='black', linestyle='none')#,label='missing')
#option 2
ax.errorbar(hist_x, hist.counts, yerr=uncertainty,  color='black', linestyle='none')
ax.stairs(hist.counts, edges=hist.bins, color='black', linewidth=1)#, label='invariant')


for idx, ex in enumerate(residual_excitations_p2):
    ax.axvline(ex,color=colors[idx])#, label=fr"$E_x$={ex:0.2f} MeV",)



# Quick
ax.set_ylabel(f"Counts/{int(granularity*1000)} keV")
ax.set_xlabel(f"{residual_2.get_latex_rep()} excitation energy (MeV)")
# ax.grid()
fig.legend()

In [ ]:
# Plot Particle 3 Excitation Spectrums with Invariant Mass and Missing Mass
# Option 1
min_bin, max_bin = -2 , 20
granularity = 0.25 # MeV
bin_size = int((max_bin - min_bin) / (granularity))
# Option 2
# bin_size = 100
# granularity = (max_bin - min_bin) / bin_size



fig, axs = plt.subplots(1,2, figsize=(16,6))
eject_kinematics = np.zeros((len(residual_excitations_p3), ngrid * 2, 2))
eject_kinematics[:, :ngrid, :] = reaction_p3.kinematic_curve(proj_energy_start,residual_excitations_p3)
eject_kinematics[:, ngrid:, :] = np.flip(reaction_p3.kinematic_curve(proj_energy_stop, residual_excitations_p3),axis=1)
colors = cmap(np.linspace(0,1,len(residual_excitations_p3)))

# Particle 1: Missing mass 
ax = axs[0]
ax.set_title("Missing Mass Technique")
grammer.add_hist1d("ex_3", bin_size, (min_bin,max_bin))
grammer.fill_hist1d("ex_3", kinematics_p3["ex"][mask])
hist  = grammer.get_hist1d("ex_3")
uncertainty = np.sqrt(np.maximum(hist.counts,1))
hist_x = hist.bins[:-1] + 0.5*hist.bin_width
#option 1
# ax.errorbar(hist_x, hist.counts, yerr=uncertainty, fmt='o', markersize=3, capsize=2,color='black', linestyle='none')#,label='missing')
#option 2
ax.errorbar(hist_x, hist.counts, yerr=uncertainty,  color='black', linestyle='none')
ax.stairs(hist.counts, edges=hist.bins, color='black', linewidth=1)
for idx, ex in enumerate(residual_excitations_p3):
    ax.axvline(ex,color=colors[idx],label=fr"$E_x$={ex:0.2f} MeV")


# Quick
ax.set_ylabel(f"Counts/{int(granularity*1000)} keV")
ax.set_xlabel(f"{residual_3.get_latex_rep()} excitation energy (MeV)")
# ax.grid()


# Particle 1: Invariant mass 
ax = axs[1]
ax.set_title("Invariant Mass Technique")
grammer.add_hist1d("ex_3_inv", bin_size, (min_bin,max_bin))
grammer.fill_hist1d("ex_3_inv", results["m12"][mask] - residual_3.mass)
hist  = grammer.get_hist1d("ex_3_inv")
uncertainty = np.sqrt(np.maximum(hist.counts,1))
hist_x = hist.bins[:-1] + 0.5*hist.bin_width
#option 1
# ax.errorbar(hist_x, hist.counts, yerr=uncertainty, fmt='o', markersize=3, capsize=2,color='black', linestyle='none')#,label='missing')
#option 2
ax.errorbar(hist_x, hist.counts, yerr=uncertainty,  color='black', linestyle='none')
ax.stairs(hist.counts, edges=hist.bins, color='black', linewidth=1)#, label='invariant')


for idx, ex in enumerate(residual_excitations_p3):
    ax.axvline(ex,color=colors[idx])#, label=fr"$E_x$={ex:0.2f} MeV",)



# Quick
ax.set_ylabel(f"Counts/{int(granularity*1000)} keV")
ax.set_xlabel(f"{residual_3.get_latex_rep()} excitation energy (MeV)")
# ax.grid()
fig.legend()

### Fusion events for 1 repeated particle 
3. In case you have 3 particles, but one particle type has a duplicate like a+ a + d, you can use the following cells. Note: Your repeated particle must be labeled as particle_1, and the duplicated type will become particle_3. 

In [ ]:
particle_1_z, particle_1_a = 1, 1 # Write repeated particle first
particle_2_z, particle_2_a = 2, 3
# particle_3 = particle_1 <--- this is the duplicated type

In [ ]:
# Construct nuclear properties 
particle_1 = nuclear_map.get_data(particle_1_z,particle_1_a)
particle_2 = nuclear_map.get_data(particle_2_z,particle_2_a)
particle_3 = particle_1
# create compound nucleus
compound_nucleus_z =particle_1_z + particle_2_z  + particle_3.Z
compound_nucleus_a = particle_1_a + particle_2_a + particle_3.A
compound_nucleus = nuclear_map.get_data(compound_nucleus_z, compound_nucleus_a)

#get residuals
residual_1 = nuclear_map.get_data(beam_z + target_z - particle_1_z, beam_a + target_a - particle_1_a)
residual_2 = nuclear_map.get_data(beam_z + target_z - particle_2_z, beam_a + target_a - particle_2_a)
residual_3 = residual_1

results = {}
results["m12"] = np.empty(0,float)
results["m13"] = np.empty(0,float)
results["m23"] = np.empty(0,float)
results["fusion_ex"]= np.empty(0,float)
results["event"] = np.empty(0,int)
results["run"] = np.empty(0,int)


# Store Physics into dictations 
dictations_items = ["brho","kinetic_energy", "polar", "az", "redchisq", "vertex_x", "vertex_y", "vertex_z","ex","polar_cm"]

kinematics_p1 = {}
kinematics_p2 = {}
kinematics_p3 = {}
for items in dictations_items:
    kinematics_p1[items] = np.empty(0,float)
    kinematics_p2[items] = np.empty(0,float)
    kinematics_p3[items] = np.empty(0,float)


# Load particle 2 particle file 
empty_files = []
data_p1 = RunSolverDataLoader(
    solver_result_path=solver_result_path,
    particle = particle_1,
    empty_files=empty_files
)
data_p2 = RunSolverDataLoader(
    solver_result_path,
    particle_2, 
    empty_files
)


for run in range(run_min, run_max): 
    # Open data frames 
    df_p1 = data_p1.load_solver(run)
    df_p2 = data_p2.load_solver(run)
    if empty_files.__contains__(run):
        continue 
    #------------------------------------------------------------------------------
    # Insert specific filters into the dataframe 
    # (Here)
    # select events inside the gates 
    events_p1, counts_p1 = np.unique(df_p1.select("orig_event").to_numpy().flatten(),return_counts=True)
    events_p1 = events_p1[counts_p1==2]

    events_p2, counts_p2 = np.unique(df_p2.select("orig_event").to_numpy().flatten(),return_counts=True)
    events_p2 = events_p2[counts_p2==1]

    events = np.intersect1d(events_p1,events_p2)

    df_repeated = df_p1.filter(pl.col("orig_event").is_in(events))
    df_p1 = df_repeated.group_by("orig_event", maintain_order=True).first()
    df_p2 = df_p2.filter(pl.col("orig_event").is_in(events))
    df_p3 = df_repeated.group_by("orig_event", maintain_order=True).last()
    #------------------------------------------------------------------------------
    # Extract reactions physics
    reaction_p1 = DataReactionKinematics(
        data=df_p1,
        projectile=beam,
        target=target,
        targetlike=particle_1,
        beamlike=residual_1, # <--- for this case, it does not really matter
        beam_energy_entrace_tpc=proj_energy_start
    )
    reaction_p2 = DataReactionKinematics(df_p2, beam, target,particle_2, residual_2, proj_energy_start)
    reaction_p3 = DataReactionKinematics(df_p3, beam, target,particle_1, residual_1, proj_energy_start)

    #close data frames 
    df_p1, df_p2, df_p3 = None, None, None

    # construct fusion vectors
    particle_1_vector = reaction_p1.construct_targetlike_vector()
    particle_2_vector = reaction_p2.construct_targetlike_vector()
    particle_3_vector = reaction_p3.construct_targetlike_vector()

    m12 = (particle_1_vector + particle_2_vector).M
    m13 = (particle_1_vector + particle_3_vector).M
    m23 = (particle_2_vector + particle_3_vector).M
    fusion_ex = (particle_1_vector+particle_2_vector+particle_3_vector).mass - compound_nucleus.mass


    # Store information 
    results["m12"] = np.append(results["m12"], m12)
    results["m13"] = np.append(results["m13"], m13)
    results["m23"] = np.append(results["m23"], m23)
    results["fusion_ex"] = np.append(results["fusion_ex"],fusion_ex)
    results["event"] = np.append(results["event"],events)
    results["run"] = np.append(results["run"],np.full(len(events),run))
    

    reaction_p1.extract_kinematics_values(kinematics_p1,target_material)
    reaction_p2.extract_kinematics_values(kinematics_p2,target_material)
    reaction_p3.extract_kinematics_values(kinematics_p3,target_material)


# remove duplicates
empty_files = sorted(list(set(empty_files)))
print("Files without data of interest", empty_files)

#### Multiple Plots 
Dalitz plot and then, excitation spectrum of the compound nucleus from fusion. 

In [ ]:
energy_threshold =  (particle_1.mass + particle_2.mass  + particle_3.mass ) - compound_nucleus.mass
print("energy threshold", energy_threshold)

In [ ]:
# Binning 
min_bin, max_bin = -2 , 60
bin_size = 100
granularity = (max_bin - min_bin) / bin_size


# Product Plots
grammer = Histogrammer()
grammer.add_hist1d("fusion_ex", bin_size, (min_bin, max_bin))
grammer.fill_hist1d("fusion_ex", results["fusion_ex"][:])
hist = grammer.get_hist1d("fusion_ex")
fig, ax = plt.subplots(1,1)
ax.stairs(hist.counts, edges=hist.bins)
ax.set_title(f"{compound_nucleus.get_latex_rep()} Fusion Evaporation ", fontsize=16)
ax.set_xlabel(r"Energy Spectrum (MeV)", fontsize=16)
ax.set_ylabel(f"Counts/{int(granularity*1000)} keV", fontsize=16)
# ax.axvline(38.27, color='blue')#, label='98 MeV (before IC)')
# # ax.axvspan(35.41,36.71,color='blue',alpha=0.2,label='AT-TPC')

ax.axvline(energy_threshold,color='black',label='threshold')
ax.legend()

In [ ]:
grammer.add_hist2d("inv_mass_2d",  (40, 40), ((4665, 4685), (7450, 7490)))

grammer.fill_hist2d("inv_mass_2d",results["m12"][:], results["m13"][:])
hist = grammer.get_hist2d("inv_mass_2d")
fig, ax = plt.subplots(1,1)
mesh = ax.pcolormesh(hist.x_bins, hist.y_bins, hist.counts)#, norm="log")

# Add resonances 
# ax.axvline(nuclear_map.get_data(4,7).mass + 4.570,color='cyan',linewidth=3, label=r'$E_x(^7\text{Be})=4.57$ MeV')
# ax.axhline((nuclear_map.get_data(3,6).mass + 2.186),color='red',linewidth=3,label=r'$E_x(^6\text{Li})=2.186$ MeV')

# Lower limit
dalitz_allowed = dalitz_plot(
    m1 = particle_1,
    m2 = particle_2,
    m3 = particle_3,
    compound = compound_nucleus,
    ex = 35.41
)

x, y_min, y_max = dalitz_allowed.values_plot(steps=1000)
# ax.plot(np.sqrt(x),np.sqrt(y_min),color='red',linewidth=3,label='exit AT-TPC')
# ax.plot(np.sqrt(x),np.sqrt(y_max),color='red',linewidth=3)


dalitz_allowed = dalitz_plot(
    m1 = particle_1,
    m2 = particle_2,
    m3 = particle_3,
    compound = compound_nucleus,
    ex = 36.71
)

x, y_min, y_max = dalitz_allowed.values_plot(steps=1000)
# ax.plot(np.sqrt(x),np.sqrt(y_min),color='blue',linewidth=3, label='entrance AT-TPC')
# ax.plot(np.sqrt(x),np.sqrt(y_max),color='blue',linewidth=3)


dalitz_allowed = dalitz_plot(
    m1 = particle_1,
    m2 = particle_2,
    m3 = particle_3,
    compound = compound_nucleus,
    ex = 39.6
)

x, y_min, y_max = dalitz_allowed.values_plot(steps=1000)
# ax.plot(np.sqrt(x),np.sqrt(y_min),color='magenta',linewidth=3, label='centroid')
# ax.plot(np.sqrt(x),np.sqrt(y_max),color='magenta',linewidth=3)

fig.colorbar(mesh, ax=ax)
ax.set_title(f"{compound_nucleus.get_latex_rep()} Fusion Evaporation ", fontsize=16)
ax.set_xlabel(f"m({particle_1.get_latex_rep()}, {particle_2.get_latex_rep()}) [MeV]", fontsize=16)
ax.set_ylabel(f"m({particle_1.get_latex_rep()}, {particle_3.get_latex_rep()}) [MeV]", fontsize=16)
fig.set_figheight(8.0)
fig.set_figwidth(11.0)
ax.legend(fontsize=16)

In [ ]:
# Save events information 
frame = {"run": results["run"], "event": results["event"]}
df = pl.DataFrame(frame)
# df.write_csv("./fusion_events.csv")

In [ ]:
# Make kinematics plots
bins = 80
ngrid = 1000 # grid for kinematic plot 
make_cuts = False 
# mask = # Add your conditions if you wanna make cuts

residual_excitations_dup = np.array([0]) #6Be
residual_excitations_p2 = np.array([11.35,16.626, 16.922]) #6Li


# Execute plots
if not make_cuts:
    mask = np.full(len(results["m12"]),True)


plt.close("all")
fig, axs = plt.subplots(1,2,figsize=(11.77,6))


ax = axs[0]
grammer.add_hist2d("kinematics_dup", (bins,bins), ((0,90),(0,70)))
grammer.fill_hist2d("kinematics_dup", np.rad2deg(kinematics_p1["polar"][mask]), kinematics_p1["kinetic_energy"][mask])
grammer.fill_hist2d("kinematics_dup", np.rad2deg(kinematics_p3["polar"][mask]), kinematics_p3["kinetic_energy"][mask])
hist = grammer.get_hist2d("kinematics_dup")
mesh = ax.pcolormesh(hist.x_bins, hist.y_bins, hist.counts, norm='log')
ax.set_title(f"Kinematics of {particle_1.get_latex_rep()}")
ax.set_xlabel(r"$\theta_\text{lab}$ (deg)")
ax.set_ylabel("Kinetic energy (MeV)")


eject_kinematics = np.zeros((len(residual_excitations_dup), ngrid * 2, 2))
eject_kinematics[:, :ngrid, :] = reaction_p1.kinematic_curve(proj_energy_start,residual_excitations_dup)
eject_kinematics[:, ngrid:, :] = np.flip(reaction_p1.kinematic_curve(proj_energy_stop, residual_excitations_dup),axis=1)

for idx, ex in enumerate(residual_excitations_dup):
    ax.plot(np.rad2deg(eject_kinematics[idx, :ngrid, 0]), eject_kinematics[idx, :ngrid, 1], label=fr"$E_x$({residual_1.get_latex_rep()})={ex:0.2f} MeV")
    ax.fill(np.rad2deg(eject_kinematics[idx, :, 0]), eject_kinematics[idx, :, 1],alpha=0.2)#, label=f"Ex={ex:02} Band")

ax.legend()

ax = axs[1]
grammer.add_hist2d("kinematics_p2", (bins,bins), ((0,180),(0,70)))
grammer.fill_hist2d("kinematics_p2", np.rad2deg(kinematics_p2["polar"][mask]), kinematics_p2["kinetic_energy"][mask])
hist = grammer.get_hist2d("kinematics_p2")
mesh = ax.pcolormesh(hist.x_bins, hist.y_bins, hist.counts, norm='log')
ax.set_title(f"Kinematics of {particle_2.get_latex_rep()}")
ax.set_xlabel(r"$\theta_\text{lab}$ (deg)")
ax.set_ylabel("Kinetic energy (MeV)")


eject_kinematics = np.zeros((len(residual_excitations_p2), ngrid * 2, 2))
eject_kinematics[:, :ngrid, :] = reaction_p2.kinematic_curve(proj_energy_start,residual_excitations_p2)
eject_kinematics[:, ngrid:, :] = np.flip(reaction_p2.kinematic_curve(proj_energy_stop, residual_excitations_p2),axis=1)

for idx, ex in enumerate(residual_excitations_p2):
    ax.plot(np.rad2deg(eject_kinematics[idx, :ngrid, 0]), eject_kinematics[idx, :ngrid, 1], label=fr"$E_x$({residual_2.get_latex_rep()})= {ex:0.2f} MeV")#,color=colors[idx])
    ax.fill(np.rad2deg(eject_kinematics[idx, :, 0]), eject_kinematics[idx, :, 1],alpha=0.2)#, color=colors[idx],)#, label=f"Ex={ex:02} Band", alpha=0.2)

ax.legend()




In [ ]:
# Plot Particle 1 and 3 Excitation Spectrums with Invariant Mass and Missing Mass
# Option 1
min_bin, max_bin = -5, 35 
granularity = 1 # MeV
bin_size = int((max_bin - min_bin) / (granularity))
# Option 2
# bin_size = 100
# granularity = (max_bin - min_bin) / bin_size



fig, axs = plt.subplots(1,2, figsize=(16,6))
eject_kinematics = np.zeros((len(residual_excitations_p1), ngrid * 2, 2))
eject_kinematics[:, :ngrid, :] = reaction_p1.kinematic_curve(proj_energy_start,residual_excitations_p1)
eject_kinematics[:, ngrid:, :] = np.flip(reaction_p1.kinematic_curve(proj_energy_stop, residual_excitations_p1),axis=1)
colors = cmap(np.linspace(0,1,len(residual_excitations_p1)))

# Particle 1: Missing mass 
ax = axs[0]
ax.set_title("Missing Mass Technique")
grammer.add_hist1d("ex_dup", bin_size, (min_bin,max_bin))
grammer.fill_hist1d("ex_dup", kinematics_p1["ex"][mask])
grammer.fill_hist1d("ex_dup", kinematics_p3["ex"][mask])
hist  = grammer.get_hist1d("ex_dup")
uncertainty = np.sqrt(np.maximum(hist.counts,1))
hist_x = hist.bins[:-1] + 0.5*hist.bin_width
#option 1
# ax.errorbar(hist_x, hist.counts, yerr=uncertainty, fmt='o', markersize=3, capsize=2,color='black', linestyle='none')#,label='missing')
#option 2
ax.errorbar(hist_x, hist.counts, yerr=uncertainty,  color='black', linestyle='none')
ax.stairs(hist.counts, edges=hist.bins, color='black', linewidth=1)
for idx, ex in enumerate(residual_excitations_dup):
    ax.axvline(ex,color=colors[idx],label=fr"$E_x$={ex:0.2f} MeV")


# Quick
ax.set_ylabel(f"Counts/{int(granularity*1000)} keV")
ax.set_xlabel(f"{residual_1.get_latex_rep()} excitation energy (MeV)")
# ax.grid()


# Particle 1: Invariant mass 
ax = axs[1]
ax.set_title("Invariant Mass Technique")
grammer.add_hist1d("ex_dup_inv", bin_size, (min_bin,max_bin))
grammer.fill_hist1d("ex_dup_inv", results["m23"][mask] - residual_1.mass)
hist  = grammer.get_hist1d("ex_dup_inv")
uncertainty = np.sqrt(np.maximum(hist.counts,1))
hist_x = hist.bins[:-1] + 0.5*hist.bin_width
#option 1
# ax.errorbar(hist_x, hist.counts, yerr=uncertainty, fmt='o', markersize=3, capsize=2,color='black', linestyle='none')#,label='missing')
#option 2
ax.errorbar(hist_x, hist.counts, yerr=uncertainty,  color='black', linestyle='none')
ax.stairs(hist.counts, edges=hist.bins, color='black', linewidth=1, label='invariant')


for idx, ex in enumerate(residual_excitations_dup):
    ax.axvline(ex,color=colors[idx])#, label=fr"$E_x$={ex:0.2f} MeV",)



# Quick
ax.set_ylabel(f"Counts/{int(granularity*1000)} keV")
ax.set_xlabel(f"{residual_1.get_latex_rep()} excitation energy (MeV)")
# ax.grid()
fig.legend()


In [ ]:
# Plot Particle 2 Excitation Spectrums with Invariant Mass and Missing Mass
# Option 1
min_bin, max_bin = -2 , 40
granularity = 1 # MeV
bin_size = int((max_bin - min_bin) / (granularity))
# Option 2
# bin_size = 100
# granularity = (max_bin - min_bin) / bin_size



fig, axs = plt.subplots(1,2, figsize=(16,6))
eject_kinematics = np.zeros((len(residual_excitations_p2), ngrid * 2, 2))
eject_kinematics[:, :ngrid, :] = reaction_p2.kinematic_curve(proj_energy_start,residual_excitations_p2)
eject_kinematics[:, ngrid:, :] = np.flip(reaction_p2.kinematic_curve(proj_energy_stop, residual_excitations_p2),axis=1)
colors = cmap(np.linspace(0,1,len(residual_excitations_p2)))

# Particle 1: Missing mass 
ax = axs[0]
ax.set_title("Missing Mass Technique")
grammer.add_hist1d("ex_2", bin_size, (min_bin,max_bin))
grammer.fill_hist1d("ex_2", kinematics_p2["ex"][mask])
hist  = grammer.get_hist1d("ex_2")
uncertainty = np.sqrt(np.maximum(hist.counts,1))
hist_x = hist.bins[:-1] + 0.5*hist.bin_width
#option 1
# ax.errorbar(hist_x, hist.counts, yerr=uncertainty, fmt='o', markersize=3, capsize=2,color='black', linestyle='none')#,label='missing')
#option 2
ax.errorbar(hist_x, hist.counts, yerr=uncertainty,  color='black', linestyle='none')
ax.stairs(hist.counts, edges=hist.bins, color='black', linewidth=1)
for idx, ex in enumerate(residual_excitations_p2):
    ax.axvline(ex,color=colors[idx],label=fr"$E_x$={ex:0.2f} MeV")


# Quick
ax.set_ylabel(f"Counts/{int(granularity*1000)} keV")
ax.set_xlabel(f"{residual_2.get_latex_rep()} excitation energy (MeV)")
# ax.grid()


# Particle 1: Invariant mass 
ax = axs[1]
ax.set_title("Invariant Mass Technique")
grammer.add_hist1d("ex_2_inv", bin_size, (min_bin,max_bin))
grammer.fill_hist1d("ex_2_inv", results["m13"][mask] - residual_2.mass)
hist  = grammer.get_hist1d("ex_2_inv")
uncertainty = np.sqrt(np.maximum(hist.counts,1))
hist_x = hist.bins[:-1] + 0.5*hist.bin_width
#option 1
# ax.errorbar(hist_x, hist.counts, yerr=uncertainty, fmt='o', markersize=3, capsize=2,color='black', linestyle='none')#,label='missing')
#option 2
ax.errorbar(hist_x, hist.counts, yerr=uncertainty,  color='black', linestyle='none')
ax.stairs(hist.counts, edges=hist.bins, color='black', linewidth=1)#, label='invariant')


for idx, ex in enumerate(residual_excitations_p2):
    ax.axvline(ex,color=colors[idx])#, label=fr"$E_x$={ex:0.2f} MeV",)



# Quick
ax.set_ylabel(f"Counts/{int(granularity*1000)} keV")
ax.set_xlabel(f"{residual_2.get_latex_rep()} excitation energy (MeV)")
# ax.grid()
fig.legend()